<p style="text-align:center; font-size: 28px;">AI in Health Technologies (2026)</p>
<p style="text-align:center; font-size: 28px;">Project: Source or clinical content?</p>
<p style="text-align:center; font-size: 20px;">Alexandra Dahlberg, Heidi Tarkiainen, Joonatan Syrjänen</p>

### Table of Contents

* [Section 1: Introduction](#section_1)
* [Section 2: Setup](#section_2)
    * [Section 2.1: Packages](#section_2_1)
    * [Section 2.2: Settings](#section_2_2)
* [Section 3: Data](#section_3)
    * [Section 3.1: Loading the data](#section_3_1)
    * [Section 3.2: Initial look](#section_3_2)
    * [Section 3.3: Diagnosis codes](#section_3_3)
    * [Section 3.4: Note length and truncation (Table 1)](#section_3_4)
* [Section 4: Embeddings](#section_4)
    * [Section 4.1: Loading FinBERT](#section_4_1)
    * [Section 4.2: All layers, mean pooled](#section_4_2)
    * [Section 4.3: Long notes in chunks](#section_4_3)
    * [Section 4.4: Check against Rami's textembed](#section_4_4)
* [Section 5: Projections](#section_5)
    * [Section 5.1: PCA](#section_5_1)
    * [Section 5.2: t-SNE](#section_5_2)
    * [Section 5.3: UMAP](#section_5_3)
* [Section 6: Separation in full and 2-D space](#section_6)
    * [Section 6.1: kNN label agreement](#section_6_1)
    * [Section 6.2: Silhouette](#section_6_2)
* [Section 7: Classifiers](#section_7)
    * [Section 7.1: Folds grouped by patient](#section_7_1)
    * [Section 7.2: Linear probe and MLP](#section_7_2)
    * [Section 7.3: Length and TF-IDF baselines](#section_7_3)
    * [Section 7.4: Chance level](#section_7_4)
* [Section 8: Synthetic data checks (Table 2)](#section_8)
    * [Section 8.1: Planted effects](#section_8_1)
    * [Section 8.2: Negative control](#section_8_2)
    * [Section 8.3: Shortcut test](#section_8_3)
* [Section 9: Q1 Source](#section_9)
    * [Section 9.1: UMAP grid (Fig. 2)](#section_9_1)
    * [Section 9.2: Separation and classifiers per label (Table 3)](#section_9_2)
* [Section 10: Q2 Diagnosis](#section_10)
    * [Section 10.1: Picture vs classifier per code (Fig. 3)](#section_10_1)
* [Section 11: Robustness and confounding](#section_11)
    * [Section 11.1: Seeds and UMAP settings](#section_11_1)
    * [Section 11.2: Confounding checks](#section_11_2)
    * [Section 11.3: Truncation check](#section_11_3)
* [Section 12: Choice of layer (Fig. 4)](#section_12)
* [Section 13: Clinical review](#section_13)
* [Section 14: Export](#section_14)

## 1. Introduction <a class="anchor" id="section_1"></a>

Something here.

## 2. Setup <a class="anchor" id="section_2"></a>

### 2.1. Packages <a class="anchor" id="section_2_1"></a>

In [7]:
# List of those that could be relevant, needs to be checked and cleaned before submission

# Data and plots
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# FinBERT
import torch
from transformers import AutoTokenizer, AutoModel

# Projections
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap  # pip install umap-learn

# kNN agreement and silhouette
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score

# Classifiers, folds and baselines
from sklearn.model_selection import GroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import roc_auc_score, balanced_accuracy_score

### 2.2. Settings <a class="anchor" id="section_2_2"></a>

In [8]:
# Synthetic or real data, everything below runs on both
data_source = "synthetic" # Change to "real" in Acamedic

if data_source == "synthetic":
    data_folder = "mock_data" # Mock tables in the repo
else:
    data_folder = "/path/to/acamedic/export" # Real export, same file names as mock_data

# Embedding model, the main one is fixed before we look at the real data
model_name = "qwen3" # "qwen3" or "finbert"

if model_name == "qwen3":
    model_path = "/path/to/Qwen3-Embedding-8B-Q6_K.gguf" # Same GGUF file locally and in Acamedic
    max_tokens = 8192 # Reads up to 32k, check the longest note in 3.4
    main_layer = None # llama.cpp only gives the pooled vector, no layers
else:
    model_path = "TurkuNLP/bert-base-finnish-cased-v1" # Local folder in Acamedic, no internet there
    max_tokens = 512 # FinBERT can't read more than this
    main_layer = 12 # The other layers are only a check

# GPU if there is one (Colab, Azure ML), otherwise CPU
if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

# Seeds
seed = 42 # Life
np.random.seed(seed)
torch.manual_seed(seed)

# Analysis settings, fixed before we look at the real data
k = 15 # Neighbours for kNN agreement
umap_seeds = [0, 1, 2, 3, 4] # Robustness check in 11.1
umap_neighbours = [5, 15, 50] # Values of n_neighbors for the same check
min_patients_per_code = 10 # Placeholder, smaller ICD codes are dropped
remove_header = False # True drops the first line (role, name, date), it gives away the hospital on its own

results_folder = "results" # Figures, tables and run log go here

## 3. Data <a class="anchor" id="section_3"></a>

### 3.1. Loading the data <a class="anchor" id="section_3_1"></a>

In [9]:
# Synthetic notes from the csv, one row per note (text + label columns)
# Like in Acamedic: needs the potilaskertomus table joined to the diagnosis table through the episode ID
# Print the number of notes and patients

# Notes and diagnoses, both tables use ; as the separator
notes = pd.read_csv(data_folder + "/Apotti_potilaskertomus.csv", sep=";")
diagnoses = pd.read_csv(data_folder + "/Apotti_diagnoosi.csv", sep=";")

# Main diagnosis per care episode (shjakso), one row each
main_diagnoses = diagnoses[diagnoses["paadiagnoosi"] == True] # Main diagnosis rows only
main_diagnoses = main_diagnoses.drop_duplicates(subset="shjakso_id") # A few episodes have the same code twice
main_diagnoses = main_diagnoses[["shjakso_id", "diagnoosi_koodi", "potilas_ika_vuosina", "potilas_sukupuoli_selite"]]

# Diagnosis, age and sex onto every note through the care episode
notes = notes.merge(main_diagnoses, on="shjakso_id", how="left")

# Labels with English names, so the rest of the notebook is easier to read
notes["patient_id"] = notes["henkilotunnus"] # Pseudonym, used to group the folds
notes["hospital"] = notes["toimipiste_selite"].str.split(", ").str[-1] # E.g. Tornisairaala, after the comma
notes["unit"] = notes["toimipiste_selite"] # Ward, outpatient clinic or theatre, with the hospital
notes["specialty"] = notes["toimipiste_eala_selite"] # Verisuonikirurgia or Anestesiologia
notes["note_type"] = notes["merkinta_valilehti_selite"] # E.g. Päivittäismerkintä, Leikkauskertomus
notes["year"] = pd.to_datetime(notes["merkinta_luonti_hetki"]).dt.year # Year the note was written
notes["diagnosis_code"] = notes["diagnoosi_koodi"] # E.g. I70.2
notes["diagnosis_group"] = notes["diagnoosi_koodi"].str[:3] # E.g. I70
notes["age"] = notes["potilas_ika_vuosina"]
notes["sex"] = notes["potilas_sukupuoli_selite"]
notes["text"] = notes["merkinta_teksti"]

# Drop the header line if the setting says so
if remove_header:
    notes["text"] = notes["text"].str.split("\n", n=1).str[1] # Everything after the first line

print("Notes: %d"
      % notes.shape[0])
print("Patients: %d"
      % notes["patient_id"].nunique())
print("Care episodes: %d"
      % notes["shjakso_id"].nunique())
print("Notes without a main diagnosis: %d"
      % notes["diagnosis_code"].isnull().sum())

Notes: 2492
Patients: 292
Care episodes: 992
Notes without a main diagnosis: 0


### 3.2. Initial look <a class="anchor" id="section_3_2"></a>

In [ ]:
# Notes per label (note type, unit, year, specialty) and notes per patient
# Check for empty or very short notes, drop them here and print how many went



### 3.3. Diagnosis codes <a class="anchor" id="section_3_3"></a>

In [11]:
# Main ICD-10 code per care episode, at the finer level (e.g. I70.2, I71.4, I65.2)
# Keep codes with at least N patients, print how many codes are left (hoping for 10+)
# Only vascular surgery notes in Acamedic, so the codes will be close to each other

### 3.4. Note length and truncation (Table 1) <a class="anchor" id="section_3_4"></a>

In [12]:
# Load the FinBERT tokenizer here already, the model itself comes in 4.1
# Tokens per note counted before truncation, median and share over 512
# Notes, patients, codes and lengths for synthetic and real data go into Table 1

## 4. Embeddings <a class="anchor" id="section_4"></a>

### 4.1. Loading FinBERT or other model (?) <a class="anchor" id="section_4_1"></a>

In [13]:
# TurkuNLP/bert-base-finnish-cased-v1 from a local folder, eval mode, weights frozen
# GPU if there is one, otherwise CPU (fine for the synthetic data)

### 4.2. All layers, mean pooled <a class="anchor" id="section_4_2"></a>

In [14]:
# output_hidden_states=True gives 13 layers (input + 12)
# Mean pooling over the real tokens using the attention mask, first 512 tokens only
# Save one array per layer to disk so we don't have to embed again, layer 12 is the main one

### 4.3. Long notes in chunks <a class="anchor" id="section_4_3"></a>

In [15]:
# Only the notes over 512 tokens: split into 512-token chunks, embed each, average
# Layer 12 only, used for the truncation check in 11.3

### 4.4. Check against Rami's textembed <a class="anchor" id="section_4_4"></a>

In [16]:
# Run Rami's textembed on the same notes (mean pooling, 512 tokens)
# Our layer 12 vectors should match his, if not something is off in the pooling

## 5. Projections <a class="anchor" id="section_5"></a>

### 5.1. PCA <a class="anchor" id="section_5_1"></a>

In [17]:
# PCA to 2-D on the layer 12 vectors, mostly as a sanity check

### 5.2. t-SNE <a class="anchor" id="section_5_2"></a>

In [18]:
# t-SNE to 2-D, cosine, perplexity 30, PCA init

### 5.3. UMAP <a class="anchor" id="section_5_3"></a>

In [19]:
# UMAP to 2-D, cosine, n_neighbors 15, min_dist 0.1, main seed
# Save the coordinates, the extra seeds and settings come in 11.1

## 6. Separation in full and 2-D space <a class="anchor" id="section_6"></a>

### 6.1. kNN label agreement <a class="anchor" id="section_6_1"></a>

In [20]:
# Helper function: for each note, share of its 15 nearest neighbours with the same label
# Cosine distance in the full 768-D space, Euclidean in 2-D
# Report it as lift over how common the label is, otherwise big classes look great by chance
# Full minus 2-D = how much the projection distorts

### 6.2. Silhouette <a class="anchor" id="section_6_2"></a>

In [21]:
# Silhouette per label in full and 2-D space, secondary, just for comparison

## 7. Classifiers <a class="anchor" id="section_7"></a>

### 7.1. Folds grouped by patient <a class="anchor" id="section_7_1"></a>

In [22]:
# 5 folds grouped by patient (GroupKFold), so the same patient is never in both train and test
# Small validation split inside the training folds for early stopping

### 7.2. Linear probe and MLP <a class="anchor" id="section_7_2"></a>

In [23]:
# Linear probe: one nn.Linear on the frozen embeddings, cross entropy, Adam
# MLP: one hidden layer, otherwise the same
# Same training loop for both, early stopping on the validation loss
# Macro AUROC and balanced accuracy per label, plus AUROC per class for Q2

### 7.3. Length and TF-IDF baselines <a class="anchor" id="section_7_3"></a>

In [24]:
# Same linear model, but trained on note length only (characters, words, tokens)
# And on TF-IDF features, fitted inside each training fold so nothing leaks from the test fold
# If these do as well as the embedding, the embedding adds little

### 7.4. Chance level <a class="anchor" id="section_7_4"></a>

In [25]:
# Shuffle the labels and rerun, gives the chance level for each label

## 8. Synthetic data checks (Table 2) <a class="anchor" id="section_8"></a>

Only for the synthetic data, since we know what we planted there.

### 8.1. Planted effects <a class="anchor" id="section_8_1"></a>

In [26]:
# Does the pipeline find the planted source and diagnosis effects?
# Diagnosis effects should come out in the order we planted them, weak to strong

### 8.2. Negative control <a class="anchor" id="section_8_2"></a>

In [27]:
# The label with no signal, we expect kNN lift and AUROC around chance

### 8.3. Shortcut test <a class="anchor" id="section_8_3"></a>

In [28]:
# Train the diagnosis classifier where diagnosis is linked to unit style, test where the link is broken
# Compare AUROC with and without the link, and check if the UMAP showed the unit clusters beforehand
# Table 2 is put together from 8.1-8.3 and saved as table2.csv

## 9. Q1 Source <a class="anchor" id="section_9"></a>

### 9.1. UMAP grid (Fig. 2) <a class="anchor" id="section_9_1"></a>

In [29]:
# Same UMAP four times, coloured by note type, unit, year and note length
# One panel per label, written out one by one
# Save as fig2_umap_grid.png

### 9.2. Separation and classifiers per label (Table 3) <a class="anchor" id="section_9_2"></a>

In [30]:
# One row per label: kNN lift full and 2-D, AUROC for embedding, TF-IDF, length and chance
# Save as table3.csv

## 10. Q2 Diagnosis <a class="anchor" id="section_10"></a>

### 10.1. Picture vs classifier per code (Fig. 3) <a class="anchor" id="section_10_1"></a>

In [31]:
# Per ICD code: 2-D kNN lift (x) against classifier AUROC (y), one dot per code
# Dot size by number of patients
# Spearman with a bootstrap CI only if there are 10+ codes, otherwise we just describe it
# Save as fig3_picture_vs_classifier.png

## 11. Robustness and confounding <a class="anchor" id="section_11"></a>

### 11.1. Seeds and UMAP settings <a class="anchor" id="section_11_1"></a>

In [32]:
# UMAP again with 5 seeds and n_neighbors 5, 15 and 50
# 2-D kNN lift for each run, report the spread and check that no conclusion flips

### 11.2. Confounding checks <a class="anchor" id="section_11_2"></a>

In [33]:
# Source labels within the same ICD code, diagnosis within the same unit and note type
# Same kNN lift and classifiers as above, just on the subsets
# Optional: train diagnosis on one note type, test on another

### 11.3. Truncation check <a class="anchor" id="section_11_3"></a>

In [34]:
# Long notes: first 512 tokens vs the chunk average from 4.3
# Does the kNN lift or AUROC change for the long notes?

## 12. Choice of layer (Fig. 4) <a class="anchor" id="section_12"></a>

In [35]:
# kNN lift in full space for every layer 0-12, one line per label
# Dashed line at layer 12, our main layer
# Optional: AlignedUMAP across the layers, stays in the notebook, not in the paper
# Save as fig4_layers.png

## 13. Clinical review <a class="anchor" id="section_13"></a>

Done inside Acamedic only.

In [36]:
# Pick a handful of notes from each visible UMAP cluster (note IDs only)
# Read them inside Acamedic and write down what separates the clusters: template, structure or content
# Nothing from here leaves Acamedic except those categories

## 14. Export <a class="anchor" id="section_14"></a>

In [37]:
# Save all figures and tables to results/ with the run date
# Quick check that no text column ends up in anything we export
# Write the run log (settings, numbers of notes, package versions) for the README